# Multimodal Attention + Positional Visualization

This notebook centralizes visualization for:
- attention masks
- Fourier positional geometry
- RoPE relative score geometry

It uses one shared mock pipeline setup and follows the same `MultimodalIO` flattening path as training.

In [ ]:
import sys
from pathlib import Path
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from omegaconf import OmegaConf


def find_repo_root(start: Path) -> Path:
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "configs" / "model" / "plai_v1.yaml").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate repository root from {start}. Expected pyproject.toml and configs/model/plai_v1.yaml in a parent directory."
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_ROOT = REPO_ROOT / "src"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from data.data_classes import FullData
from models.components.multimodal_io import MultimodalIO
from models.components.positional_encoding import RotaryEmbedding
from utils.constants import (
    AUDIO_TOKEN_FPS,
    KEYBOARD_FPS,
    MODALITY_SHAPES,
    MOUSE_FPS,
    VIDEO_FPS,
)

print("Resolved REPO_ROOT:", REPO_ROOT)

In [ ]:
# Edit this single config block to control all plots consistently.
VIZ_CONFIG: Dict[str, Any] = {
    "batch_size": 1,
    "num_frames": 2,
    "include_video": True,
    "device": "auto",  # auto | cuda | cpu
    "seed": 0,
    "mask_type": "token_level",  # token_level | dataframe_level | no_mask
    "positional_encoding_type": "fourier",  # fourier | trainable | rope
    "fourier_num_bands": 32,
    "fourier_concat_pos": True,
    "fourier_sine_only": False,
    "temporal_fourier_min_resolution": 0.02,
    "temporal_fourier_max_resolution": 50.0,
    "video_spatial_fourier_min_resolution": [0.0833, 0.05],
    "video_spatial_fourier_max_resolution": [0.5, 0.5],
    "rope_theta": 50000,
    "rope_max_spatial_freq": 0.15,
    "rope_probe_dim": 64,
    "rope_probe_mrope_section": [16, 8, 8],
    "rope_temporal_scale_factor": 100.0,
}

torch.manual_seed(int(VIZ_CONFIG["seed"]))

In [ ]:
def build_mmio_cfg_from_training(viz_cfg: Dict[str, Any]) -> Dict[str, Any]:
    model_cfg = OmegaConf.load(REPO_ROOT / "configs" / "model" / "plai_v1.yaml")
    mmio_cfg = OmegaConf.to_container(model_cfg.multimodal_io, resolve=True)
    mmio_cfg.pop("_target_", None)

    if not bool(viz_cfg["include_video"]):
        mmio_cfg["context_modalities"] = [m for m in mmio_cfg["context_modalities"] if m != "video"]
        mmio_cfg["target_modalities"] = [m for m in mmio_cfg["target_modalities"] if m != "video"]

    overrides = {
        "mask_type": viz_cfg["mask_type"],
        "positional_encoding_type": viz_cfg["positional_encoding_type"],
        "fourier_num_bands": int(viz_cfg["fourier_num_bands"]),
        "fourier_concat_pos": bool(viz_cfg["fourier_concat_pos"]),
        "fourier_sine_only": bool(viz_cfg["fourier_sine_only"]),
        "temporal_fourier_min_resolution": float(viz_cfg["temporal_fourier_min_resolution"]),
        "temporal_fourier_max_resolution": float(viz_cfg["temporal_fourier_max_resolution"]),
        "video_spatial_fourier_min_resolution": list(viz_cfg["video_spatial_fourier_min_resolution"]),
        "video_spatial_fourier_max_resolution": list(viz_cfg["video_spatial_fourier_max_resolution"]),
        "rope_temporal_scale_factor": float(viz_cfg["rope_temporal_scale_factor"]),
    }
    mmio_cfg.update(overrides)
    return mmio_cfg


def make_mock_fulldata(
    batch_size: int,
    num_frames: int,
    start_df_index: int,
    include_video: bool,
    device: torch.device,
) -> FullData:
    metadata = [
        {
            "player_name": f"player_{i}",
            "player_gender": "unknown",
            "player_skill_level": "mid",
        }
        for i in range(batch_size)
    ]

    batch = {
        "audio_speak": torch.randn(batch_size, num_frames, *MODALITY_SHAPES["audio_speak"], device=device),
        "audio_hear": torch.randn(batch_size, num_frames, *MODALITY_SHAPES["audio_hear"], device=device),
        "key_press": torch.randn(batch_size, num_frames, *MODALITY_SHAPES["key_press"], device=device),
        "mouse_movement": torch.randn(batch_size, num_frames, *MODALITY_SHAPES["mouse_movement"], device=device),
        "metadata": metadata,
        "dataframe_indices": torch.arange(start_df_index, start_df_index + num_frames, dtype=torch.long, device=device)
        .unsqueeze(0)
        .expand(batch_size, -1),
    }

    if include_video:
        batch["video"] = torch.randn(batch_size, num_frames, *MODALITY_SHAPES["video"], device=device)
    else:
        batch["video"] = None

    return FullData(batch=batch)


def dense_mask_from_block(mask: Any, target_time: torch.Tensor) -> torch.Tensor:
    seq_len = int(target_time.shape[1])
    if mask is None:
        return torch.ones((seq_len, seq_len), dtype=torch.bool)

    if hasattr(mask, "mask_mod"):
        b_idx = torch.zeros((1, 1, 1, 1), dtype=torch.long)
        h_idx = torch.zeros((1, 1, 1, 1), dtype=torch.long)
        q_idx = torch.arange(seq_len, dtype=torch.long).view(1, 1, -1, 1)
        kv_idx = torch.arange(seq_len, dtype=torch.long).view(1, 1, 1, -1)
        return mask.mask_mod(b_idx, h_idx, q_idx, kv_idx)[0, 0]

    dense = mask.to_dense() if hasattr(mask, "to_dense") else mask
    if dense.ndim == 4:
        return dense[0, 0].to(torch.bool).cpu()
    if dense.ndim == 3:
        return dense[0].to(torch.bool).cpu()
    return dense.to(torch.bool).cpu()


def per_timestep_modality_layout(
    active_modality_names: List[str],
    modality_shapes: Dict[str, Tuple[int, int]],
) -> List[Tuple[str, int]]:
    return [(name, int(modality_shapes[name][1])) for name in active_modality_names]


def build_modality_token_indices(
    active_modality_names: List[str],
    modality_shapes: Dict[str, Tuple[int, int]],
) -> Dict[str, List[int]]:
    timesteps = int(modality_shapes[active_modality_names[0]][0])
    idx: Dict[str, List[int]] = {name: [] for name in active_modality_names}
    cursor = 0
    for _ in range(timesteps):
        for name in active_modality_names:
            token_len = int(modality_shapes[name][1])
            idx[name].extend(list(range(cursor, cursor + token_len)))
            cursor += token_len
    return idx


def encode_raw(module: Any, pos: torch.Tensor, nyquist_fps: float | None = None) -> torch.Tensor:
    raw_module = module.base_encoding if hasattr(module, "base_encoding") else module
    if hasattr(raw_module, "encode_raw"):
        out = raw_module.encode_raw(batch_size=1, pos=pos, nyquist_fps=nyquist_fps)
    else:
        out = raw_module(batch_size=1, pos=pos, nyquist_fps=nyquist_fps)
    return out.reshape(pos.shape[0], -1)

In [ ]:
requested_device = str(VIZ_CONFIG["device"]).lower()
if requested_device == "auto":
    resolved_device = "cuda" if torch.cuda.is_available() else "cpu"
elif requested_device == "cuda" and not torch.cuda.is_available():
    print("CUDA requested but unavailable; falling back to CPU.")
    resolved_device = "cpu"
else:
    resolved_device = requested_device

device = torch.device(resolved_device)
mmio_cfg = build_mmio_cfg_from_training(VIZ_CONFIG)
mmio = MultimodalIO(**mmio_cfg).to(device)

num_frames = int(VIZ_CONFIG["num_frames"])
batch_size = int(VIZ_CONFIG["batch_size"])
include_video = bool(VIZ_CONFIG["include_video"])

# Context uses dataframe indices -2, -1 by default when num_frames=2.
context_fd = make_mock_fulldata(
    batch_size=batch_size,
    num_frames=num_frames,
    start_df_index=-num_frames,
    include_video=include_video,
    device=device,
)
# Target uses dataframe indices 0, 1 by default when num_frames=2.
target_fd = make_mock_fulldata(
    batch_size=batch_size,
    num_frames=num_frames,
    start_df_index=0,
    include_video=include_video,
    device=device,
)

context_in = mmio.fulldata_to_context_embedder_input(context_fd)
decoder_in = mmio.fulldata_to_moe_decoder_input(target_fd, mask_type=VIZ_CONFIG["mask_type"])

print("resolved device:", resolved_device)
print("context dataframe_indices:", context_fd.dataframe_indices[0].tolist())
print("target dataframe_indices:", target_fd.dataframe_indices[0].tolist())
print("active modalities:", decoder_in["active_modality_names"])
print("x_flat shape:", tuple(decoder_in["x_flat"].shape))
print("target_time shape:", tuple(decoder_in["target_time"].shape))


In [ ]:
# Visualize x_flat token structure: show first 2 and last 2 tokens per modality + ellipsis for middle
modality_shapes = decoder_in["modality_shapes"]
active_modalities = decoder_in["active_modality_names"]
num_frames_viz = int(VIZ_CONFIG["num_frames"])

# Color map for modalities
colors = plt.cm.tab10(range(len(active_modalities)))
mod_to_color = {mod: colors[i] for i, mod in enumerate(active_modalities)}

# Pre-calculate token ranges for each frame
frame_token_ranges = []
cursor = 0
for frame_idx in range(num_frames_viz):
    frame_start = cursor
    for mod_name in active_modalities:
        cursor += int(modality_shapes[mod_name][1])
    frame_token_ranges.append((frame_start, cursor))

# Build rows: show first 2, then last 2 tokens per modality, with ellipsis in between
fig, axes = plt.subplots(num_frames_viz, 1, figsize=(14, num_frames_viz * 1.2), squeeze=False)
axes = axes.flatten()

for frame_idx in range(num_frames_viz):
    ax = axes[frame_idx]
    
    frame_start, frame_end = frame_token_ranges[frame_idx]
    token_plot_pos = 0
    
    for mod_idx, mod_name in enumerate(active_modalities):
        num_tokens_in_mod = int(modality_shapes[mod_name][1])
        num_show_first = 2
        num_show_last = 2
        
        # Helper function to draw a single token
        def draw_token(local_idx, pos_x):
            abs_token_idx = frame_start + sum(int(modality_shapes[m][1]) for m in active_modalities[:mod_idx]) + local_idx
            
            # Draw token block
            rect = plt.Rectangle((pos_x, 0), 0.8, 1, 
                                 facecolor=mod_to_color[mod_name], alpha=0.5, 
                                 edgecolor="black", linewidth=1.5)
            ax.add_patch(rect)
            
            # Get position label with proper precision
            if mod_name == "video":
                pos = decoder_in["target_rope_pos"][0, abs_token_idx, :].cpu().tolist()
                label = f"t:{pos[0]:.3f}\nx:{pos[1]:.2f}\ny:{pos[2]:.2f}"
            else:
                t = decoder_in["target_time"][0, abs_token_idx].cpu().item()
                label = f"t:{t:.3f}"
            
            ax.text(pos_x + 0.4, 0.5, label, ha="center", va="center", 
                   fontsize=7, weight="bold", 
                   bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.9, edgecolor="none"))
            
            return pos_x + 1.0
        
        # Draw first N tokens
        for local_idx in range(min(num_show_first, num_tokens_in_mod)):
            token_plot_pos = draw_token(local_idx, token_plot_pos)
        
        # Show ellipsis and last N tokens if there are more than (first + last)
        if num_tokens_in_mod > num_show_first + num_show_last:
            remaining = num_tokens_in_mod - num_show_first - num_show_last
            ax.text(token_plot_pos + 0.2, 0.5, f"...\n({remaining} more)", 
                   ha="center", va="center", fontsize=8, weight="bold", style="italic",
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray", alpha=0.7))
            token_plot_pos += 0.8
            
            # Draw last N tokens
            for i in range(num_show_last):
                local_idx = num_tokens_in_mod - num_show_last + i
                token_plot_pos = draw_token(local_idx, token_plot_pos)
        elif num_tokens_in_mod > num_show_first:
            # If we have more than first but less than first+last, just show the remaining ones
            for local_idx in range(num_show_first, num_tokens_in_mod):
                token_plot_pos = draw_token(local_idx, token_plot_pos)
        
        # Add modality separator
        if mod_idx < len(active_modalities) - 1:
            ax.axvline(token_plot_pos + 0.1, color="gray", linewidth=2, linestyle="--", alpha=0.5)
            token_plot_pos += 0.3
    
    # Formatting
    ax.set_xlim(-0.3, token_plot_pos + 0.5)
    ax.set_ylim(-0.15, 1.15)
    ax.set_aspect("auto")
    ax.set_yticks([])
    ax.set_xticks([])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.spines["left"].set_visible(False)
    
    # Title with dataframe index
    ax.set_ylabel(f"df={frame_idx}", fontsize=11, weight="bold", style="italic")
    
    # Add modality names as x-axis-like labels
    token_plot_pos = 0
    for mod_idx, mod_name in enumerate(active_modalities):
        num_tokens_in_mod = int(modality_shapes[mod_name][1])
        num_show_first = 2
        num_show_last = 2
        
        # Calculate display width for this modality
        shown_count = min(num_show_first, num_tokens_in_mod)
        if num_tokens_in_mod > num_show_first + num_show_last:
            shown_count += 1  # ellipsis
            shown_count += num_show_last
        else:
            shown_count = min(num_tokens_in_mod, num_show_first + num_show_last)
        
        shown_width = shown_count * 1.0 if shown_count > 0 else 0.5
        
        mid_x = token_plot_pos + shown_width / 2
        ax.text(mid_x, -0.08, mod_name, ha="center", va="top", fontsize=8, 
               weight="bold", color=mod_to_color[mod_name])
        
        token_plot_pos += shown_width
        if mod_idx < len(active_modalities) - 1:
            token_plot_pos += 0.3

# Add legend
legend_patches = [plt.Rectangle((0, 0), 1, 1, facecolor=mod_to_color[mod], alpha=0.5, 
                               edgecolor="black", linewidth=1.5, label=mod) 
                  for mod in active_modalities]

plt.suptitle(f"x_flat Token Structure — seq_len={decoder_in['x_flat'].shape[1]}, embedding_dim={decoder_in['x_flat'].shape[-1]})", 
            fontsize=11, weight="bold", y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# Print summary
total_tokens = decoder_in['x_flat'].shape[1]
print(f"\nToken structure summary:")
print(f"  Total sequence length: {total_tokens} tokens")
print(f"  Embedding dimension: {decoder_in['x_flat'].shape[-1]}")
print(f"  Dataframes: {num_frames_viz}")
print(f"  Modalities per frame: {len(active_modalities)}")
print(f"\nTokens per modality per dataframe:")
for frame_idx in range(num_frames_viz):
    print(f"  df={frame_idx}:", end="")
    for mod_name in active_modalities:
        num_tokens = int(modality_shapes[mod_name][1])
        print(f" {mod_name}({num_tokens})", end="")
    print()


In [ ]:
# Plot all three mask types, with and without video
mask_types = ["token_level", "dataframe_level", "no_mask"]
video_configs = [True, False]
plot_configs = [(m, v) for m in mask_types for v in video_configs]

# One plot per row for readability
fig, axes = plt.subplots(len(plot_configs), 1, figsize=(16, 10 * len(plot_configs)), dpi=120)
if len(plot_configs) == 1:
    axes = [axes]

for idx, (mask_type, include_vid) in enumerate(plot_configs):
    # Build config for this specific mask/video combination
    viz_cfg_local = VIZ_CONFIG.copy()
    viz_cfg_local["mask_type"] = mask_type
    viz_cfg_local["include_video"] = include_vid

    # Build MultimodalIO with local config
    mmio_cfg_local = build_mmio_cfg_from_training(viz_cfg_local)
    mmio_local = MultimodalIO(**mmio_cfg_local).to(device)

    # Create mock data
    target_fd_local = make_mock_fulldata(
        batch_size=1,
        num_frames=int(viz_cfg_local["num_frames"]),
        start_df_index=0,
        include_video=include_vid,
        device=device,
    )

    # Generate decoder input with the local mask type
    decoder_in_local = mmio_local.fulldata_to_moe_decoder_input(target_fd_local, mask_type=mask_type)

    # Get dense mask
    dense_mask_local = dense_mask_from_block(decoder_in_local["block_mask"], decoder_in_local["target_time"])
    dense_np = dense_mask_local.detach().cpu().numpy().astype(bool)

    active_mods_local = decoder_in_local["active_modality_names"]
    layout_local = per_timestep_modality_layout(active_mods_local, decoder_in_local["modality_shapes"])
    tokens_per_df = sum(count for _, count in layout_local)
    num_df_local = int(viz_cfg_local["num_frames"])
    seq_len = dense_np.shape[0]

    # Stable per-plot modality palette
    palette = plt.cm.tab10(range(len(active_mods_local)))
    mod_to_color_local = {mod: palette[i][:3] for i, mod in enumerate(active_mods_local)}

    # Build token -> modality map over full flattened sequence
    token_to_mod = []
    for _ in range(num_df_local):
        for mod_name, count in layout_local:
            token_to_mod.extend([mod_name] * int(count))
    if len(token_to_mod) != seq_len:
        raise ValueError(f"token_to_mod length {len(token_to_mod)} != seq_len {seq_len}")

    # Render custom RGB matrix in [query, key] indexing:
    # - same-modality attention: solid modality color
    # - cross-modality attention: checker-stripe of query/key modality colors
    # - disallowed attention: white
    rgb_qk = torch.ones((seq_len, seq_len, 3), dtype=torch.float32)
    for q in range(seq_len):
        q_mod = token_to_mod[q]
        q_color = mod_to_color_local[q_mod]
        for k in range(seq_len):
            if not dense_np[q, k]:
                continue
            k_mod = token_to_mod[k]
            k_color = mod_to_color_local[k_mod]
            if q_mod == k_mod:
                rgb_qk[q, k] = torch.tensor(q_color)
            else:
                # Checker stripe between query-modality and key-modality colors
                rgb_qk[q, k] = torch.tensor(q_color if ((q + k) % 2 == 0) else k_color)

    # Plot with query on x-axis and key on y-axis
    # imshow convention is [y, x], so transpose from [query, key] -> [key, query]
    ax = axes[idx]
    ax.imshow(rgb_qk.transpose(0, 1).numpy(), origin="lower", interpolation="nearest")

    # Add dataframe boundaries (same positions on both axes)
    for step in range(1, num_df_local):
        boundary = step * tokens_per_df
        ax.axvline(boundary - 0.5, color="black", linewidth=1.4)
        ax.axhline(boundary - 0.5, color="black", linewidth=1.4)

    # Add modality boundaries within each dataframe
    offsets_local = []
    running = 0
    for _, count in layout_local[:-1]:
        running += int(count)
        offsets_local.append(running)
    for step in range(num_df_local):
        base = step * tokens_per_df
        for off in offsets_local:
            boundary = base + off
            ax.axvline(boundary - 0.5, color="gray", linewidth=0.8, linestyle="--", alpha=0.9)
            ax.axhline(boundary - 0.5, color="gray", linewidth=0.8, linestyle="--", alpha=0.9)

    video_str = "with video" if include_vid else "no video"
    ax.set_title(f"{mask_type} ({video_str})")
    ax.set_xlabel("Query (target token index)")
    ax.set_ylabel("Key (target token index)")

    # Explicitly state target vs context for this matrix
    # ax.text(
    #     0.01,
    #     0.99,
    #     "Target-only mask (context tokens are not in this matrix)",
    #     transform=ax.transAxes,
    #     ha="left",
    #     va="top",
    #     fontsize=8,
    #     bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="black", alpha=0.9),
    # )

    # Legend: modality solids only
    modality_handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor=mod_to_color_local[m], edgecolor="black", linewidth=1.0, label=m)
        for m in active_mods_local
    ]

    ax.legend(
        handles=modality_handles,
        title="Modality Colors",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
        fontsize=8,
        title_fontsize=9,
    )

plt.tight_layout()
plt.show()


In [ ]:
# Fourier positional encoding diagnostics (distance-only)
"""
Edit LOCAL_FOURIER_CONFIG below to experiment locally without changing training config.
Only keys set to non-None values will override VIZ_CONFIG for this cell.
"""

LOCAL_FOURIER_CONFIG = {
    "mask_type": "no_mask",
    "fourier_num_bands": 64,
    "fourier_concat_pos": True,
    "fourier_sine_only": False,
    "temporal_fourier_min_resolution": 0.1,
    "temporal_fourier_max_resolution": 4.0,
    "video_spatial_fourier_min_resolution": [0.02, 0.0333],
    "video_spatial_fourier_max_resolution": [0.12, 0.20],
}

viz_cfg_fourier = VIZ_CONFIG.copy()
viz_cfg_fourier["positional_encoding_type"] = "fourier"
for key, value in LOCAL_FOURIER_CONFIG.items():
    if value is not None:
        viz_cfg_fourier[key] = value

mmio_cfg_fourier = build_mmio_cfg_from_training(viz_cfg_fourier)
mmio_fourier = MultimodalIO(**mmio_cfg_fourier).to(device)

target_fd_fourier = make_mock_fulldata(
    batch_size=1,
    num_frames=int(viz_cfg_fourier["num_frames"]),
    start_df_index=0,
    include_video=bool(viz_cfg_fourier["include_video"]),
    device=device,
 )

decoder_in_fourier = mmio_fourier.fulldata_to_moe_decoder_input(
    target_fd_fourier, mask_type=str(viz_cfg_fourier["mask_type"])
)

print("Local Fourier overrides:", {k: v for k, v in LOCAL_FOURIER_CONFIG.items() if v is not None})
print(
    "Effective Fourier config:",
    {
        "mask_type": viz_cfg_fourier["mask_type"],
        "fourier_num_bands": viz_cfg_fourier["fourier_num_bands"],
        "fourier_concat_pos": viz_cfg_fourier["fourier_concat_pos"],
        "fourier_sine_only": viz_cfg_fourier["fourier_sine_only"],
        "temporal_fourier_min_resolution": viz_cfg_fourier["temporal_fourier_min_resolution"],
        "temporal_fourier_max_resolution": viz_cfg_fourier["temporal_fourier_max_resolution"],
        "video_spatial_fourier_min_resolution": viz_cfg_fourier["video_spatial_fourier_min_resolution"],
        "video_spatial_fourier_max_resolution": viz_cfg_fourier["video_spatial_fourier_max_resolution"],
    },
)

if viz_cfg_fourier["positional_encoding_type"] == "fourier":
    fps_by_modality = {
        "audio_speak": float(AUDIO_TOKEN_FPS),
        "audio_hear": float(AUDIO_TOKEN_FPS),
        "key_press": float(KEYBOARD_FPS),
        "mouse_movement": float(MOUSE_FPS),
        "video": float(VIDEO_FPS),
    }

    active_fourier = decoder_in_fourier["active_modality_names"]
    modality_shapes_fourier = decoder_in_fourier["modality_shapes"]
    token_indices_fourier = build_modality_token_indices(active_fourier, modality_shapes_fourier)

    # Modality-wise Fourier geometry (one plot per row)
    fig, axes = plt.subplots(len(active_fourier), 1, figsize=(16, 10 * len(active_fourier)), dpi=120)
    if len(active_fourier) == 1:
        axes = [axes]

    dist_by_modality = {}
    for ax, name in zip(axes, active_fourier):
        idx = torch.tensor(token_indices_fourier[name], dtype=torch.long, device=decoder_in_fourier["target_time"].device)
        if name == "video":
            coords = decoder_in_fourier["target_rope_pos"][0, idx, :]
            feats = encode_raw(mmio_fourier.video_pos_encoding, coords, nyquist_fps=float(VIDEO_FPS))
        else:
            coords = decoder_in_fourier["target_time"][0, idx].reshape(-1, 1)
            feats = encode_raw(mmio_fourier.seq_pos_encoding, coords, nyquist_fps=fps_by_modality[name])

        sim = torch.matmul(F.normalize(feats, p=2, dim=-1), F.normalize(feats, p=2, dim=-1).transpose(0, 1))
        dist = 1.0 - sim
        dist_np = dist.detach().cpu().numpy()
        dist_by_modality[name] = dist_np

        im = ax.imshow(dist_np, aspect="auto", cmap="magma", vmin=0.0, origin="lower")
        ax.set_title(f"Fourier temporal distance: {name}")
        ax.set_xlabel("Key index")
        ax.set_ylabel("Query index")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

    # Combined view over full flattened sequence.
    # Rule: cross-modality pairs compare on shared temporal features only;
    # video-video pairs compare on temporal + video spatial residual.
    seq_len_total = int(decoder_in_fourier["target_time"].shape[1])
    active_mods_local = decoder_in_fourier["active_modality_names"]

    # Shared temporal basis for ALL tokens so cross-modality temporal distances are meaningful.
    full_time = decoder_in_fourier["target_time"][0].reshape(-1, 1)
    temporal_shared = encode_raw(mmio_fourier.seq_pos_encoding, full_time, nyquist_fps=float(MOUSE_FPS)).to(torch.float32)

    # Build token->modality map in the same flattened order as attention mask plotting.
    token_to_mod = []
    for _ in range(int(viz_cfg_fourier["num_frames"])):
        for mod_name, count in per_timestep_modality_layout(active_mods_local, decoder_in_fourier["modality_shapes"]):
            token_to_mod.extend([mod_name] * int(count))
    if len(token_to_mod) != seq_len_total:
        raise ValueError(f"token_to_mod length {len(token_to_mod)} != seq_len_total {seq_len_total}")

    # Optional video-only spatial residual (video full spatiotemporal minus pure temporal-in-video-space).
    if "video" in active_mods_local:
        video_idx = torch.tensor(token_indices_fourier["video"], dtype=torch.long, device=decoder_in_fourier["target_time"].device)
        video_coords = decoder_in_fourier["target_rope_pos"][0, video_idx, :]

        video_full = encode_raw(mmio_fourier.video_pos_encoding, video_coords, nyquist_fps=float(VIDEO_FPS)).to(torch.float32)

        temporal_only_coords = video_coords.clone()
        temporal_only_coords[:, 1] = 0.0
        temporal_only_coords[:, 2] = 0.0
        video_temporal_in_video_space = encode_raw(
            mmio_fourier.video_pos_encoding, temporal_only_coords, nyquist_fps=float(VIDEO_FPS)
        ).to(torch.float32)

        video_spatial_residual = video_full - video_temporal_in_video_space

        spatial_block = torch.zeros((seq_len_total, int(video_spatial_residual.shape[-1])), dtype=torch.float32, device=temporal_shared.device)
        spatial_block[video_idx] = video_spatial_residual

        full_feats = torch.cat([temporal_shared, spatial_block], dim=-1)
    else:
        full_feats = temporal_shared

    # Temporal-only similarity for all pairs
    temp_norm = F.normalize(temporal_shared, p=2, dim=-1)
    sim_temporal = torch.matmul(temp_norm, temp_norm.transpose(0, 1))

    if "video" in active_mods_local:
        # Full similarity includes video spatial residual
        full_norm = F.normalize(full_feats, p=2, dim=-1)
        sim_full = torch.matmul(full_norm, full_norm.transpose(0, 1))

        # Use full sim only for video-video pairs; otherwise temporal-only sim
        is_video = torch.tensor([m == "video" for m in token_to_mod], dtype=torch.bool, device=sim_temporal.device)
        vv_mask = is_video[:, None] & is_video[None, :]
        sim_combined = sim_temporal.clone()
        sim_combined[vv_mask] = sim_full[vv_mask]
    else:
        sim_combined = sim_temporal

    full_dist = (1.0 - sim_combined).detach().cpu().numpy()

    layout_local = per_timestep_modality_layout(active_mods_local, decoder_in_fourier["modality_shapes"])
    tokens_per_df = sum(count for _, count in layout_local)
    num_df_local = int(viz_cfg_fourier["num_frames"])
    fig_all, ax_all = plt.subplots(1, 1, figsize=(14, 12), dpi=130)
    im_all = ax_all.imshow(full_dist, origin="lower", interpolation="nearest", cmap="magma", vmin=0.0)
    fig_all.colorbar(im_all, ax=ax_all, fraction=0.046, pad=0.04)
    ax_all.set_title("Fourier distance (all modalities; cross-modality=temporal, video-video=spatiotemporal)")
    ax_all.set_xlabel("Key (target token index)")
    ax_all.set_ylabel("Query (target token index)")

    # Dataframe boundaries (same pattern as attention mask cell)
    for step in range(1, num_df_local):
        boundary = step * tokens_per_df
        ax_all.axvline(boundary - 0.5, color="black", linewidth=1.4)
        ax_all.axhline(boundary - 0.5, color="black", linewidth=1.4)

    # Modality boundaries within each dataframe
    offsets_local = []
    running = 0
    for _, count in layout_local[:-1]:
        running += int(count)
        offsets_local.append(running)
    for step in range(num_df_local):
        base = step * tokens_per_df
        for off in offsets_local:
            boundary = base + off
            ax_all.axvline(boundary - 0.5, color="gray", linewidth=0.8, linestyle="--", alpha=0.9)
            ax_all.axhline(boundary - 0.5, color="gray", linewidth=0.8, linestyle="--", alpha=0.9)

    plt.tight_layout()
    plt.show()

    # Extra diagnostics for video: crop + zoom with sampled token (t, x, y) labels
    if "video" in dist_by_modality:
        video_idx = torch.tensor(
            token_indices_fourier["video"], dtype=torch.long, device=decoder_in_fourier["target_time"].device
        )
        video_coords = decoder_in_fourier["target_rope_pos"][0, video_idx, :].detach().cpu().numpy()
        video_dist = dist_by_modality["video"]
        n = int(video_dist.shape[0])

        # Select a few representative crops: beginning, middle, end
        crop_size = min(30, n)
        starts = [0, max(0, n // 2 - crop_size // 2), max(0, n - crop_size)]

        # Remove duplicates while preserving order
        seen = set()
        unique_starts = []
        for s in starts:
            if s not in seen:
                unique_starts.append(s)
                seen.add(s)

        fig_zoom, axes_zoom = plt.subplots(len(unique_starts), 1, figsize=(16, 6 * len(unique_starts)), dpi=140)
        if len(unique_starts) == 1:
            axes_zoom = [axes_zoom]

        for ax, start in zip(axes_zoom, unique_starts):
            end = min(n, start + crop_size)
            crop = video_dist[start:end, start:end]

            im = ax.imshow(crop, origin="lower", interpolation="nearest", cmap="magma", vmin=0.0)
            ax.set_title(f"Video Fourier distance zoom: tokens [{start}:{end})")
            ax.set_xlabel("Key index (local crop)")
            ax.set_ylabel("Query index (local crop)")
            fig_zoom.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

            # Add sparse (t, x, y) labels on axes ticks (not all tokens)
            tick_count = min(6, max(2, end - start))
            tick_positions = torch.linspace(0, end - start - 1, steps=tick_count).round().long().tolist()
            tick_labels = []
            for p in tick_positions:
                g = start + p
                t, x, y = video_coords[g]
                tick_labels.append(f"{g}\n(t={t:.3f},x={x:.2f},y={y:.2f})")

            ax.set_xticks(tick_positions)
            ax.set_yticks(tick_positions)
            ax.set_xticklabels(tick_labels, fontsize=7, rotation=30, ha="right")
            ax.set_yticklabels(tick_labels, fontsize=7)

        plt.tight_layout()
        plt.show()

        print("Video zoom note: labels show global token index and sampled (t, x, y) coordinates.")

In [ ]:
# VideoRoPE attention-score diagnostics using the same flattened decoder sequence
# Improvements in this cell:
# 1) stronger dataframe/modality boundaries,
# 2) explicit modality labels on top/right axes,
# 3) three scenarios: all modalities, video-only, and non-video-only,
# 4) one additional cross-modal plot:
#       query = video tokens only
#       key   = all non-video modalities

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from typing import Any, Dict, List, Tuple

LOCAL_ROPE_CONFIG = {
    "constant_value": 1.0,
    "rope_probe_dim": 64,
    "rope_theta": 50000,
    # optional local overrides; set to None to use model config
    "rope_temporal_scale_factor": None,
    "rope_max_spatial_freq": None,
    "rope_mrope_section": [16, 8, 8],
}

# Read canonical RoPE hyperparameters from model config to match training behavior.
model_cfg_rope = OmegaConf.load(REPO_ROOT / "configs" / "model" / "plai_v1.yaml")
moe_rope_cfg = model_cfg_rope.moe_decoder
mmio_rope_cfg = model_cfg_rope.multimodal_io

rope_probe_dim = int(LOCAL_ROPE_CONFIG["rope_probe_dim"])
constant_value = float(LOCAL_ROPE_CONFIG["constant_value"])

rope_theta = float(
    LOCAL_ROPE_CONFIG["rope_theta"]
    if LOCAL_ROPE_CONFIG["rope_theta"] is not None
    else moe_rope_cfg.rope_theta
)
rope_temporal_scale_factor = float(
    LOCAL_ROPE_CONFIG["rope_temporal_scale_factor"]
    if LOCAL_ROPE_CONFIG["rope_temporal_scale_factor"] is not None
    else mmio_rope_cfg.rope_temporal_scale_factor
)
rope_max_spatial_freq = float(
    LOCAL_ROPE_CONFIG["rope_max_spatial_freq"]
    if LOCAL_ROPE_CONFIG["rope_max_spatial_freq"] is not None
    else moe_rope_cfg.rope_max_spatial_freq
)
rope_mrope_section = [
    int(v) for v in (
        LOCAL_ROPE_CONFIG["rope_mrope_section"]
        if LOCAL_ROPE_CONFIG["rope_mrope_section"] is not None
        else moe_rope_cfg.rope_mrope_section
    )
]

if rope_probe_dim % 2 != 0:
    raise ValueError(f"rope_probe_dim must be even, got {rope_probe_dim}")
if sum(rope_mrope_section) != (rope_probe_dim // 2):
    raise ValueError(
        f"sum(rope_mrope_section) must equal rope_probe_dim//2 ({rope_probe_dim // 2}), got {rope_mrope_section}"
    )

rope = RotaryEmbedding(
    dim=rope_probe_dim,
    theta=rope_theta,
    max_spatial_freq=rope_max_spatial_freq,
    mrope_section=rope_mrope_section,
).to(device)

scenario_specs = [
    {
        "name": "all_modalities",
        "title": "All modalities",
        "include_video": True,
        "target_modalities": None,
        "context_modalities": None,
    },
    {
        "name": "video_only",
        "title": "Video only",
        "include_video": True,
        "target_modalities": ["video"],
        "context_modalities": ["video"],
    },
    {
        "name": "non_video_only",
        "title": "All except video",
        "include_video": False,
        "target_modalities": None,
        "context_modalities": None,
    },
]


def build_decoder_bundle_for_scenario(spec: Dict[str, Any]) -> Tuple[Dict[str, Any], MultimodalIO, FullData]:
    viz_cfg_local = VIZ_CONFIG.copy()
    viz_cfg_local["include_video"] = bool(spec["include_video"])
    viz_cfg_local["positional_encoding_type"] = "rope"
    viz_cfg_local["rope_temporal_scale_factor"] = float(
        LOCAL_ROPE_CONFIG["rope_temporal_scale_factor"]
        if LOCAL_ROPE_CONFIG["rope_temporal_scale_factor"] is not None
        else VIZ_CONFIG["rope_temporal_scale_factor"]
    )

    mmio_cfg_local = build_mmio_cfg_from_training(viz_cfg_local)
    if spec["target_modalities"] is not None:
        mmio_cfg_local["target_modalities"] = list(spec["target_modalities"])
    if spec["context_modalities"] is not None:
        mmio_cfg_local["context_modalities"] = list(spec["context_modalities"])

    mmio_local = MultimodalIO(**mmio_cfg_local).to(device)
    target_fd_local = make_mock_fulldata(
        batch_size=1,
        num_frames=int(viz_cfg_local["num_frames"]),
        start_df_index=0,
        include_video=bool(spec["include_video"]),
        device=device,
    )
    decoder_local = mmio_local.fulldata_to_moe_decoder_input(
        target_fd_local, mask_type=str(viz_cfg_local["mask_type"])
    )
    return decoder_local, mmio_local, target_fd_local


def build_modality_centers(layout_local: List[Tuple[str, int]], num_df_local: int) -> Tuple[List[float], List[str]]:
    centers: List[float] = []
    labels: List[str] = []
    tokens_per_df_local = sum(int(count) for _, count in layout_local)
    for df_idx in range(num_df_local):
        base = df_idx * tokens_per_df_local
        offset = 0
        for mod_name, count in layout_local:
            count = int(count)
            center = base + offset + (count - 1) / 2.0
            centers.append(center)
            labels.append(f"df{df_idx}:{mod_name}")
            offset += count
    return centers, labels


scenario_bundles: Dict[str, Dict[str, Any]] = {}
results = []

for spec in scenario_specs:
    decoder_local, mmio_local, target_fd_local = build_decoder_bundle_for_scenario(spec)
    scenario_bundles[spec["name"]] = {
        "decoder": decoder_local,
        "mmio": mmio_local,
        "target_fd": target_fd_local,
    }

    seq_len = int(decoder_local["x_flat"].shape[1])

    q_const = torch.full(
        (1, 1, seq_len, rope_probe_dim),
        fill_value=constant_value,
        dtype=decoder_local["x_flat"].dtype,
        device=decoder_local["x_flat"].device,
    )
    k_const = torch.full(
        (1, 1, seq_len, rope_probe_dim),
        fill_value=constant_value,
        dtype=decoder_local["x_flat"].dtype,
        device=decoder_local["x_flat"].device,
    )

    rope_pos = decoder_local["target_rope_pos"]
    if rope_pos.ndim != 3:
        raise ValueError(f"decoder_in['target_rope_pos'] must be [B, N, C], got {tuple(rope_pos.shape)}")
    rope_pos = rope_pos[:1].to(device=decoder_local["x_flat"].device, dtype=decoder_local["x_flat"].dtype)

    q_rot = rope.apply_multimodal_rotary_pos_emb(q_const, rope_pos, seq_dim=-2)
    k_rot = rope.apply_multimodal_rotary_pos_emb(k_const, rope_pos, seq_dim=-2)

    q_tok = F.normalize(q_rot[0, 0], p=2, dim=-1)
    k_tok = F.normalize(k_rot[0, 0], p=2, dim=-1)
    qk_scores = torch.matmul(q_tok, k_tok.transpose(0, 1))
    qk_np = qk_scores.detach().cpu().numpy()

    layout_local = per_timestep_modality_layout(
        decoder_local["active_modality_names"],
        decoder_local["modality_shapes"],
    )
    tokens_per_df = sum(int(count) for _, count in layout_local)
    first_mod = decoder_local["active_modality_names"][0]
    num_df_local = int(decoder_local["modality_shapes"][first_mod][0])

    vmin = max(0.0, float(qk_np.min()))
    vmax = float(qk_np.max())
    if vmax <= vmin:
        vmax = vmin + 1e-6

    results.append(
        {
            "spec": spec,
            "qk_np": qk_np,
            "seq_len": seq_len,
            "layout_local": layout_local,
            "tokens_per_df": tokens_per_df,
            "num_df_local": num_df_local,
            "vmin": vmin,
            "vmax": vmax,
        }
    )

# Main three plots
fig, axes = plt.subplots(len(results), 1, figsize=(20, 12 * len(results)), dpi=130)
if len(results) == 1:
    axes = [axes]

for ax, item in zip(axes, results):
    qk_np = item["qk_np"]
    seq_len = int(item["seq_len"])
    layout_local = item["layout_local"]
    tokens_per_df = int(item["tokens_per_df"])
    num_df_local = int(item["num_df_local"])
    vmin = float(item["vmin"])
    vmax = float(item["vmax"])
    title = str(item["spec"]["title"])

    im = ax.imshow(
        qk_np,
        origin="lower",  # Bottom-left corner is token index 0 for both query/key.
        interpolation="nearest",
        cmap="magma",
        vmin=vmin,
        vmax=vmax,
    )
    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cb.set_label("Normalized q·k score", rotation=270, labelpad=14)

    # Strong dataframe boundaries.
    for step in range(1, num_df_local):
        boundary = step * tokens_per_df
        ax.axvline(boundary - 0.5, color="black", linewidth=2.8)
        ax.axhline(boundary - 0.5, color="black", linewidth=2.8)

    centers, labels = build_modality_centers(layout_local, num_df_local)
    ax_bottom = ax.secondary_xaxis("bottom")
    ax_bottom.set_xticks(centers)
    ax_bottom.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax_bottom.set_xlabel("Modality segments (per dataframe)", fontsize=10)

    ax_left = ax.secondary_yaxis("left")
    ax_left.set_yticks(centers)
    ax_left.set_yticklabels(labels, fontsize=8)
    ax_left.set_ylabel("Modality segments (per dataframe)", fontsize=10)

    ax.set_title(f"VideoRoPE q·k heatmap ({title})", fontsize=14, weight="bold")
    ax.set_xlabel("Key token index (decoder sequence)", fontsize=11)
    ax.set_ylabel("Query token index (decoder sequence)", fontsize=11)

plt.tight_layout()
plt.show()

# Additional plot: query = video tokens only, key = all non-video modalities
bundle_all = scenario_bundles["all_modalities"]
decoder_all = bundle_all["decoder"]
mmio_all = bundle_all["mmio"]
target_fd_all = bundle_all["target_fd"]

result_all = next(item for item in results if item["spec"]["name"] == "all_modalities")
qk_np_all = result_all["qk_np"]
layout_all = result_all["layout_local"]
tokens_per_df_all = int(result_all["tokens_per_df"])
num_df_all = int(result_all["num_df_local"])

# Build row/col index subsets in flattened decoder order
video_row_indices: List[int] = []
non_video_col_indices: List[int] = []

key_centers: List[float] = []
key_labels: List[str] = []

video_offset_within_df = 0
tmp_offset = 0
for mod_name, count in layout_all:
    if mod_name == "video":
        video_offset_within_df = tmp_offset
        break
    tmp_offset += int(count)

local_key_cursor = 0
for df_idx in range(num_df_all):
    base = df_idx * tokens_per_df_all
    offset = 0
    for mod_name, count in layout_all:
        count = int(count)
        global_segment = list(range(base + offset, base + offset + count))
        if mod_name == "video":
            video_row_indices.extend(global_segment)
        else:
            non_video_col_indices.extend(global_segment)
            key_centers.append(local_key_cursor + (count - 1) / 2.0)
            key_labels.append(f"df{df_idx}:{mod_name}")
            local_key_cursor += count
        offset += count

video_row_indices = np.array(video_row_indices, dtype=np.int64)
non_video_col_indices = np.array(non_video_col_indices, dtype=np.int64)

cross_np = qk_np_all[np.ix_(video_row_indices, non_video_col_indices)]

# Build sampled y-axis labels for video tokens
rope_pos_all = decoder_all["target_rope_pos"][:1].detach().cpu()

video_tokens_per_df = int(decoder_all["modality_shapes"]["video"][1])

f_video = int(MODALITY_SHAPES["video"][0])
gh = int(MODALITY_SHAPES["video"][2]) // int(mmio_all.patch_h)
gw = int(MODALITY_SHAPES["video"][3]) // int(mmio_all.patch_w)
patches_per_frame = gh * gw

tick_count_y = min(8, max(2, cross_np.shape[0]))
tick_positions_y = np.linspace(0, cross_np.shape[0] - 1, tick_count_y).round().astype(int).tolist()

y_tick_labels: List[str] = []
for p in tick_positions_y:
    g = int(video_row_indices[p])

    df_idx = g // tokens_per_df_all
    local_in_df = g - (df_idx * tokens_per_df_all + video_offset_within_df)
    frame_idx = local_in_df // patches_per_frame
    patch_idx = local_in_df % patches_per_frame
    h_idx = patch_idx // gw
    w_idx = patch_idx % gw

    rt, rx, ry = rope_pos_all[0, g, :].tolist()
    y_tick_labels.append(
        f"g={g}\nvideo\ndf={df_idx}, f={frame_idx}, h={h_idx}, w={w_idx}\nrope=({rt:.2f}, {rx:.2f}, {ry:.2f})"
    )

fig_cross, ax_cross = plt.subplots(1, 1, figsize=(22, 12), dpi=140)

im_cross = ax_cross.imshow(
    cross_np,
    origin="lower",
    interpolation="nearest",
    cmap="magma",
    aspect="auto",
)
cb_cross = fig_cross.colorbar(im_cross, ax=ax_cross, fraction=0.03, pad=0.02)
cb_cross.set_label("Normalized q·k score", rotation=270, labelpad=14)

# X-axis: centers of non-video modality segments
ax_cross.set_xticks(key_centers)
ax_cross.set_xticklabels(key_labels, rotation=45, ha="right", fontsize=8)

# Y-axis: sampled video token labels
ax_cross.set_yticks(tick_positions_y)
ax_cross.set_yticklabels(y_tick_labels, fontsize=8)

# Dataframe boundaries on query side (video rows)
for step in range(1, num_df_all):
    ax_cross.axhline(step * video_tokens_per_df - 0.5, color="black", linewidth=2.0)

# Dataframe boundaries on key side (non-video cols)
non_video_tokens_per_df = sum(int(count) for mod_name, count in layout_all if mod_name != "video")
for step in range(1, num_df_all):
    ax_cross.axvline(step * non_video_tokens_per_df - 0.5, color="black", linewidth=2.0)

ax_cross.set_title(
    "VideoRoPE q·k heatmap: query = video tokens, key = all non-video modalities",
    fontsize=14,
    weight="bold",
)
ax_cross.set_xlabel("Key tokens (all modalities except video)", fontsize=11)
ax_cross.set_ylabel("Query tokens (video only)", fontsize=11)

plt.tight_layout()
plt.show()

print("VideoRoPE diagnostics summary:")
for item in results:
    spec = item["spec"]
    print(
        f"  {spec['title']}: seq_len={item['seq_len']}, "
        f"score range=[{item['vmin']:.6f}, {item['vmax']:.6f}], "
        f"modalities={item['layout_local']}"
    )
print(f"  constant q/k value: {constant_value}")
print(f"  rope_probe_dim: {rope_probe_dim}")
print(f"  rope_theta: {rope_theta}")
print(f"  rope_temporal_scale_factor: {rope_temporal_scale_factor}")
print(f"  rope_mrope_section: {rope_mrope_section}")

In [ ]:
# VideoRoPE-style diagonal layout test (diagonal only)
# -----------------------------------------------------------------------------
# What this tests:
#   - raw notebook decoder coords stay unchanged in model code
#   - we build an alternate "diagonal layout" rope_pos just for visualization
#   - video tokens: frame center sits on diagonal, spatial patches are offsets around it
#   - non-video / 1D tokens: mapped to the diagonal as [tau, tau, tau]
#
# This cell does NOT modify source code.
#
# Assumes earlier notebook setup already defined:
#   device, REPO_ROOT, OmegaConf, RotaryEmbedding,
#   MultimodalIO, make_mock_fulldata,
#   build_mmio_cfg_from_training, build_modality_centers,
#   per_timestep_modality_layout, MODALITY_SHAPES, VIZ_CONFIG

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ----------------------------
# Local override knobs
# ----------------------------
LOCAL_DIAG_TEST_CONFIG = {
    "constant_value": 1.0,
    "rope_probe_dim": 64,
    "rope_theta": 50000,
    # optional local override; set to None to use VIZ_CONFIG / model config path
    "rope_temporal_scale_factor": 100.0,
    "rope_max_spatial_freq": None,
    "rope_mrope_section": [16, 8, 8],
}

# ----------------------------
# Load model config defaults
# ----------------------------
model_cfg_rope = OmegaConf.load(REPO_ROOT / "configs" / "model" / "plai_v1.yaml")
moe_rope_cfg = model_cfg_rope.moe_decoder

rope_probe_dim = int(LOCAL_DIAG_TEST_CONFIG["rope_probe_dim"])
constant_value = float(LOCAL_DIAG_TEST_CONFIG["constant_value"])
rope_theta = float(LOCAL_DIAG_TEST_CONFIG["rope_theta"])
rope_max_spatial_freq = float(
    moe_rope_cfg.rope_max_spatial_freq
    if LOCAL_DIAG_TEST_CONFIG["rope_max_spatial_freq"] is None
    else LOCAL_DIAG_TEST_CONFIG["rope_max_spatial_freq"]
)
rope_mrope_section = [
    int(v) for v in (
        moe_rope_cfg.rope_mrope_section
        if LOCAL_DIAG_TEST_CONFIG["rope_mrope_section"] is None
        else LOCAL_DIAG_TEST_CONFIG["rope_mrope_section"]
    )
]

if rope_probe_dim % 2 != 0:
    raise ValueError(f"rope_probe_dim must be even, got {rope_probe_dim}")
if sum(rope_mrope_section) != (rope_probe_dim // 2):
    raise ValueError(
        f"sum(rope_mrope_section) must equal rope_probe_dim//2 ({rope_probe_dim // 2}), "
        f"got {rope_mrope_section}"
    )

rope = RotaryEmbedding(
    dim=rope_probe_dim,
    theta=rope_theta,
    max_spatial_freq=rope_max_spatial_freq,
    mrope_section=rope_mrope_section,
).to(device)

# ----------------------------
# Scenario definitions
# ----------------------------
diag_scenario_specs = [
    {
        "name": "all_modalities",
        "title": "All modalities",
        "include_video": True,
        "target_modalities": None,
        "context_modalities": None,
    },
    {
        "name": "video_only",
        "title": "Video only",
        "include_video": True,
        "target_modalities": ["video"],
        "context_modalities": ["video"],
    },
    {
        "name": "non_video_only",
        "title": "All except video",
        "include_video": False,
        "target_modalities": None,
        "context_modalities": None,
    },
]

# ----------------------------
# Helpers
# ----------------------------
def build_video_patch_lookup(mmio_local):
    """
    Returns:
      f_video, gh, gw, patches_per_frame
    """
    video_shape = MODALITY_SHAPES["video"]   # [F, C, H, W]
    f_video = int(video_shape[0])
    gh = int(video_shape[2]) // int(mmio_local.patch_h)
    gw = int(video_shape[3]) // int(mmio_local.patch_w)
    patches_per_frame = gh * gw
    return f_video, gh, gw, patches_per_frame


def build_decoder_bundle_for_diag_scenario(spec):
    viz_cfg_local = VIZ_CONFIG.copy()
    viz_cfg_local["include_video"] = bool(spec["include_video"])
    viz_cfg_local["positional_encoding_type"] = "rope"
    viz_cfg_local["rope_temporal_scale_factor"] = float(
        LOCAL_DIAG_TEST_CONFIG["rope_temporal_scale_factor"]
        if LOCAL_DIAG_TEST_CONFIG["rope_temporal_scale_factor"] is not None
        else VIZ_CONFIG["rope_temporal_scale_factor"]
    )

    mmio_cfg_local = build_mmio_cfg_from_training(viz_cfg_local)
    if spec["target_modalities"] is not None:
        mmio_cfg_local["target_modalities"] = list(spec["target_modalities"])
    if spec["context_modalities"] is not None:
        mmio_cfg_local["context_modalities"] = list(spec["context_modalities"])

    mmio_local = MultimodalIO(**mmio_cfg_local).to(device)
    target_fd_local = make_mock_fulldata(
        batch_size=1,
        num_frames=int(viz_cfg_local["num_frames"]),
        start_df_index=0,
        include_video=bool(spec["include_video"]),
        device=device,
    )
    decoder_local = mmio_local.fulldata_to_moe_decoder_input(
        target_fd_local,
        mask_type=str(viz_cfg_local["mask_type"]),
    )
    return decoder_local, mmio_local, target_fd_local


def make_diagonal_layout_rope_pos(decoder_local, mmio_local):
    """
    Build notebook-side VideoRoPE-style diagonal coords from raw target_rope_pos.

    Raw input from your current code:
      - video tokens:     [tau, x, y]
      - non-video tokens: [tau, tau, tau]

    Diagonal-layout override used here:
      - video tokens:     [tau, tau + (x - cx), tau + (y - cy)]
      - non-video tokens: [tau, tau, tau]

    where cx, cy are frame-center patch coordinates.
    """
    rope_pos_raw = decoder_local["target_rope_pos"][:1].detach().clone()
    rope_pos_diag = rope_pos_raw.clone()

    active_modality_names = list(decoder_local["active_modality_names"])
    modality_shapes = decoder_local["modality_shapes"]
    layout_local = per_timestep_modality_layout(active_modality_names, modality_shapes)

    first_mod = active_modality_names[0]
    num_df_local = int(modality_shapes[first_mod][0])
    tokens_per_df = sum(int(count) for _, count in layout_local)

    _, gh, gw, _ = build_video_patch_lookup(mmio_local)
    cx = (gh - 1) / 2.0
    cy = (gw - 1) / 2.0

    modality_offsets = {}
    running = 0
    for mod_name, count in layout_local:
        modality_offsets[mod_name] = (running, int(count))
        running += int(count)

    for df_idx in range(num_df_local):
        df_base = df_idx * tokens_per_df

        for mod_name in active_modality_names:
            local_offset, local_count = modality_offsets[mod_name]
            token_indices = torch.arange(
                df_base + local_offset,
                df_base + local_offset + local_count,
                device=rope_pos_diag.device,
                dtype=torch.long,
            )

            raw = rope_pos_raw[0, token_indices, :]
            tau = raw[:, 0]

            if mod_name == "video":
                x = raw[:, 1]
                y = raw[:, 2]

                rope_pos_diag[0, token_indices, 0] = tau
                rope_pos_diag[0, token_indices, 1] = tau + (x - cx)
                rope_pos_diag[0, token_indices, 2] = tau + (y - cy)
            else:
                rope_pos_diag[0, token_indices, 0] = tau
                rope_pos_diag[0, token_indices, 1] = tau
                rope_pos_diag[0, token_indices, 2] = tau

    return rope_pos_diag


def compute_qk_heatmap(rope_module, rope_pos, seq_len, dtype, dev, constant_value=1.0):
    q_const = torch.full((1, 1, seq_len, rope_probe_dim), constant_value, dtype=dtype, device=dev)
    k_const = torch.full((1, 1, seq_len, rope_probe_dim), constant_value, dtype=dtype, device=dev)

    q_rot = rope_module.apply_multimodal_rotary_pos_emb(q_const, rope_pos, seq_dim=-2)
    k_rot = rope_module.apply_multimodal_rotary_pos_emb(k_const, rope_pos, seq_dim=-2)

    q_tok = F.normalize(q_rot[0, 0], p=2, dim=-1)
    k_tok = F.normalize(k_rot[0, 0], p=2, dim=-1)
    qk_scores = torch.matmul(q_tok, k_tok.transpose(0, 1))
    return qk_scores.detach().cpu().numpy()


def plot_heatmap(ax, fig, qk_np, layout_local, num_df_local, title):
    tokens_per_df = sum(int(count) for _, count in layout_local)
    vmin = max(0.0, float(qk_np.min()))
    vmax = float(qk_np.max())
    if vmax <= vmin:
        vmax = vmin + 1e-6

    im = ax.imshow(
        qk_np,
        origin="lower",
        interpolation="nearest",
        cmap="magma",
        vmin=vmin,
        vmax=vmax,
    )
    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cb.set_label("Normalized q·k score", rotation=270, labelpad=14)

    for step in range(1, num_df_local):
        boundary = step * tokens_per_df
        ax.axvline(boundary - 0.5, color="black", linewidth=2.5)
        ax.axhline(boundary - 0.5, color="black", linewidth=2.5)

    centers, labels = build_modality_centers(layout_local, num_df_local)
    ax_bottom = ax.secondary_xaxis("bottom")
    ax_bottom.set_xticks(centers)
    ax_bottom.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax_bottom.set_xlabel("Modality segments (per dataframe)", fontsize=10)

    ax_left = ax.secondary_yaxis("left")
    ax_left.set_yticks(centers)
    ax_left.set_yticklabels(labels, fontsize=8)
    ax_left.set_ylabel("Modality segments (per dataframe)", fontsize=10)

    ax.set_title(title, fontsize=13, weight="bold")
    ax.set_xlabel("Key token index", fontsize=11)
    ax.set_ylabel("Query token index", fontsize=11)


def plot_cross_modal_video_vs_nonvideo(ax, fig, qk_np, decoder_local, mmio_local, title):
    layout_local = per_timestep_modality_layout(
        decoder_local["active_modality_names"],
        decoder_local["modality_shapes"],
    )
    first_mod = decoder_local["active_modality_names"][0]
    num_df_local = int(decoder_local["modality_shapes"][first_mod][0])
    tokens_per_df = sum(int(count) for _, count in layout_local)

    video_row_indices = []
    non_video_col_indices = []
    key_centers = []
    key_labels = []

    video_offset_within_df = 0
    tmp_offset = 0
    for mod_name, count in layout_local:
        if mod_name == "video":
            video_offset_within_df = tmp_offset
            break
        tmp_offset += int(count)

    local_key_cursor = 0
    for df_idx in range(num_df_local):
        base = df_idx * tokens_per_df
        offset = 0
        for mod_name, count in layout_local:
            count = int(count)
            global_segment = list(range(base + offset, base + offset + count))
            if mod_name == "video":
                video_row_indices.extend(global_segment)
            else:
                non_video_col_indices.extend(global_segment)
                key_centers.append(local_key_cursor + (count - 1) / 2.0)
                key_labels.append(f"df{df_idx}:{mod_name}")
                local_key_cursor += count
            offset += count

    video_row_indices = np.array(video_row_indices, dtype=np.int64)
    non_video_col_indices = np.array(non_video_col_indices, dtype=np.int64)
    cross_np = qk_np[np.ix_(video_row_indices, non_video_col_indices)]

    _, gh, gw, patches_per_frame = build_video_patch_lookup(mmio_local)
    video_tokens_per_df = int(decoder_local["modality_shapes"]["video"][1])

    tick_count_y = min(8, max(2, cross_np.shape[0]))
    tick_positions_y = np.linspace(0, cross_np.shape[0] - 1, tick_count_y).round().astype(int).tolist()

    y_tick_labels = []
    rope_pos_all = decoder_local["target_rope_pos"][:1].detach().cpu()
    for p in tick_positions_y:
        g = int(video_row_indices[p])
        df_idx = g // tokens_per_df
        local_in_df = g - (df_idx * tokens_per_df + video_offset_within_df)
        frame_idx = local_in_df // patches_per_frame
        patch_idx = local_in_df % patches_per_frame
        h_idx = patch_idx // gw
        w_idx = patch_idx % gw
        rt, rx, ry = rope_pos_all[0, g, :].tolist()
        y_tick_labels.append(
            f"g={g}\nvideo\ndf={df_idx}, f={frame_idx}, h={h_idx}, w={w_idx}\nrope=({rt:.2f}, {rx:.2f}, {ry:.2f})"
        )

    im = ax.imshow(
        cross_np,
        origin="lower",
        interpolation="nearest",
        cmap="magma",
        aspect="auto",
    )
    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cb.set_label("Normalized q·k score", rotation=270, labelpad=14)

    ax.set_xticks(key_centers)
    ax.set_xticklabels(key_labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(tick_positions_y)
    ax.set_yticklabels(y_tick_labels, fontsize=8)

    non_video_tokens_per_df = sum(int(count) for mod_name, count in layout_local if mod_name != "video")
    for step in range(1, num_df_local):
        ax.axhline(step * video_tokens_per_df - 0.5, color="black", linewidth=2.0)
        ax.axvline(step * non_video_tokens_per_df - 0.5, color="black", linewidth=2.0)

    ax.set_title(title, fontsize=13, weight="bold")
    ax.set_xlabel("Key tokens (all non-video modalities)", fontsize=11)
    ax.set_ylabel("Query tokens (video only)", fontsize=11)


def run_diag_scenario(spec):
    decoder_local, mmio_local, target_fd_local = build_decoder_bundle_for_diag_scenario(spec)

    rope_pos_diag = make_diagonal_layout_rope_pos(decoder_local, mmio_local)

    seq_len = int(decoder_local["x_flat"].shape[1])
    layout_local = per_timestep_modality_layout(
        decoder_local["active_modality_names"],
        decoder_local["modality_shapes"],
    )
    first_mod = decoder_local["active_modality_names"][0]
    num_df_local = int(decoder_local["modality_shapes"][first_mod][0])

    qk_diag = compute_qk_heatmap(
        rope_module=rope,
        rope_pos=rope_pos_diag.to(device=decoder_local["x_flat"].device, dtype=decoder_local["x_flat"].dtype),
        seq_len=seq_len,
        dtype=decoder_local["x_flat"].dtype,
        dev=decoder_local["x_flat"].device,
        constant_value=constant_value,
    )

    decoder_diag_view = dict(decoder_local)
    decoder_diag_view["target_rope_pos"] = rope_pos_diag

    return {
        "spec": spec,
        "decoder": decoder_local,
        "decoder_diag_view": decoder_diag_view,
        "mmio": mmio_local,
        "target_fd": target_fd_local,
        "rope_pos_diag": rope_pos_diag,
        "qk_diag": qk_diag,
        "layout": layout_local,
        "num_df": num_df_local,
        "seq_len": seq_len,
    }


# ----------------------------
# Run all three scenarios
# ----------------------------
diag_results = {spec["name"]: run_diag_scenario(spec) for spec in diag_scenario_specs}

all_res = diag_results["all_modalities"]
video_res = diag_results["video_only"]
nonvideo_res = diag_results["non_video_only"]

# ----------------------------
# Plot diagonal-only results
# ----------------------------
fig, axes = plt.subplots(2, 2, figsize=(24, 22), dpi=140)

plot_heatmap(
    axes[0, 0],
    fig,
    all_res["qk_diag"],
    all_res["layout"],
    all_res["num_df"],
    title="Diagonal layout: full sequence (all modalities)",
)

plot_heatmap(
    axes[0, 1],
    fig,
    video_res["qk_diag"],
    video_res["layout"],
    video_res["num_df"],
    title="Diagonal layout: full sequence (video only)",
)

plot_heatmap(
    axes[1, 0],
    fig,
    nonvideo_res["qk_diag"],
    nonvideo_res["layout"],
    nonvideo_res["num_df"],
    title="Diagonal layout: full sequence (all except video)",
)

plot_cross_modal_video_vs_nonvideo(
    axes[1, 1],
    fig,
    all_res["qk_diag"],
    all_res["decoder_diag_view"],
    all_res["mmio"],
    title="Diagonal layout: query = video, key = all non-video",
)

plt.tight_layout()
plt.show()

# ----------------------------
# Print a few example coords
# ----------------------------
print("Diagonal-layout test summary")
print(f"  rope_probe_dim: {rope_probe_dim}")
print(f"  rope_theta: {rope_theta}")
print(f"  rope_temporal_scale_factor: {LOCAL_DIAG_TEST_CONFIG['rope_temporal_scale_factor']}")
print(f"  rope_mrope_section: {rope_mrope_section}")
print()

print("All-modalities scenario:")
print(f"  seq_len: {all_res['seq_len']}")
print("  First 12 diagonal-layout rope_pos rows:")
print(all_res["rope_pos_diag"][0, :12].detach().cpu())
print()

print("Video-only scenario:")
print(f"  seq_len: {video_res['seq_len']}")
print("  First 12 diagonal-layout rope_pos rows:")
print(video_res["rope_pos_diag"][0, :12].detach().cpu())
print()

print("Non-video-only scenario:")
print(f"  seq_len: {nonvideo_res['seq_len']}")
print("  First 12 diagonal-layout rope_pos rows:")
print(nonvideo_res["rope_pos_diag"][0, :12].detach().cpu())

In [ ]:
# Paper-style 3D position plot:
# compare current RoPE-input coords vs notebook diagonal override
#
# Important:
#   - "current" below means decoder_local["target_rope_pos"], i.e. the actual coord tensor
#     that would be passed into RoPE in the current model path.
#   - this already includes rope_temporal_scale_factor if your upstream builder applied it.
#   - the diagonal override is built FROM those current RoPE-input coords, so the scaled tau
#     stays intact and you can verify the effect numerically.

import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# ----------------------------
# Local knobs
# ----------------------------
LOCAL_3D_POS_CONFIG = {
    "include_video": True,
    "target_modalities": None,
    "context_modalities": None,
    "rope_temporal_scale_factor": 100.0,   # set None to fall back to VIZ_CONFIG
    "max_points_per_modality": None,       # e.g. 300 for lighter plots
    "points_per_modality_to_log": 6,
    "elev": 24,
    "azim": -63,
    "figsize": (18, 8),
    "dpi": 140,
    "line_alpha": 0.80,
    "point_alpha": 0.95,
    "point_size": 18,
    "linewidth": 1.6,
    "show_lines": True,
    "show_points": True,
}

# ----------------------------
# Build one decoder bundle
# ----------------------------
viz_cfg_local = VIZ_CONFIG.copy()
viz_cfg_local["include_video"] = bool(LOCAL_3D_POS_CONFIG["include_video"])
viz_cfg_local["positional_encoding_type"] = "rope"
if LOCAL_3D_POS_CONFIG["rope_temporal_scale_factor"] is not None:
    viz_cfg_local["rope_temporal_scale_factor"] = float(LOCAL_3D_POS_CONFIG["rope_temporal_scale_factor"])

mmio_cfg_local = build_mmio_cfg_from_training(viz_cfg_local)
if LOCAL_3D_POS_CONFIG["target_modalities"] is not None:
    mmio_cfg_local["target_modalities"] = list(LOCAL_3D_POS_CONFIG["target_modalities"])
if LOCAL_3D_POS_CONFIG["context_modalities"] is not None:
    mmio_cfg_local["context_modalities"] = list(LOCAL_3D_POS_CONFIG["context_modalities"])

mmio_local = MultimodalIO(**mmio_cfg_local).to(device)

target_fd_local = make_mock_fulldata(
    batch_size=1,
    num_frames=int(viz_cfg_local["num_frames"]),
    start_df_index=0,
    include_video=bool(viz_cfg_local["include_video"]),
    device=device,
)

decoder_local = mmio_local.fulldata_to_moe_decoder_input(
    target_fd_local,
    mask_type=str(viz_cfg_local["mask_type"]),
)

# ----------------------------
# Helpers
# ----------------------------
def build_video_patch_lookup(mmio_local):
    video_shape = MODALITY_SHAPES["video"]   # [F, C, H, W]
    f_video = int(video_shape[0])
    gh = int(video_shape[2]) // int(mmio_local.patch_h)
    gw = int(video_shape[3]) // int(mmio_local.patch_w)
    patches_per_frame = gh * gw
    return f_video, gh, gw, patches_per_frame


def make_diagonal_layout_rope_pos_from_current(decoder_local, mmio_local):
    """
    Build notebook-side VideoRoPE-style diagonal coords from the CURRENT RoPE-input coords.

    Input:
      decoder_local["target_rope_pos"] = actual coord tensor passed to RoPE by the model path.
      For current code, this should already contain the scaled temporal coordinate.

    Current coord convention:
      - video tokens:     [tau_scaled, x, y]
      - non-video tokens: [tau_scaled, tau_scaled, tau_scaled]

    Diagonal override:
      - video tokens:     [tau_scaled, tau_scaled + (x - cx), tau_scaled + (y - cy)]
      - non-video tokens: [tau_scaled, tau_scaled, tau_scaled]
    """
    rope_pos_current = decoder_local["target_rope_pos"][:1].detach().clone()
    rope_pos_diag = rope_pos_current.clone()

    active_modality_names = list(decoder_local["active_modality_names"])
    modality_shapes = decoder_local["modality_shapes"]
    layout_local = per_timestep_modality_layout(active_modality_names, modality_shapes)

    first_mod = active_modality_names[0]
    num_df_local = int(modality_shapes[first_mod][0])
    tokens_per_df = sum(int(count) for _, count in layout_local)

    _, gh, gw, _ = build_video_patch_lookup(mmio_local)
    cx = (gh - 1) / 2.0
    cy = (gw - 1) / 2.0

    modality_offsets = {}
    running = 0
    for mod_name, count in layout_local:
        modality_offsets[mod_name] = (running, int(count))
        running += int(count)

    for df_idx in range(num_df_local):
        df_base = df_idx * tokens_per_df

        for mod_name in active_modality_names:
            local_offset, local_count = modality_offsets[mod_name]
            token_indices = torch.arange(
                df_base + local_offset,
                df_base + local_offset + local_count,
                device=rope_pos_diag.device,
                dtype=torch.long,
            )

            cur = rope_pos_current[0, token_indices, :]
            tau = cur[:, 0]

            if mod_name == "video":
                x = cur[:, 1]
                y = cur[:, 2]
                rope_pos_diag[0, token_indices, 0] = tau
                rope_pos_diag[0, token_indices, 1] = tau + (x - cx)
                rope_pos_diag[0, token_indices, 2] = tau + (y - cy)
            else:
                rope_pos_diag[0, token_indices, 0] = tau
                rope_pos_diag[0, token_indices, 1] = tau
                rope_pos_diag[0, token_indices, 2] = tau

    return rope_pos_current, rope_pos_diag


def get_modality_indices_map(decoder_local):
    """
    Return dict: modality_name -> np.array(global token indices for that modality across all dataframes)
    preserving decoder sequence order.
    """
    active_modality_names = list(decoder_local["active_modality_names"])
    modality_shapes = decoder_local["modality_shapes"]
    layout_local = per_timestep_modality_layout(active_modality_names, modality_shapes)

    first_mod = active_modality_names[0]
    num_df_local = int(modality_shapes[first_mod][0])
    tokens_per_df = sum(int(count) for _, count in layout_local)

    modality_offsets = {}
    running = 0
    for mod_name, count in layout_local:
        modality_offsets[mod_name] = (running, int(count))
        running += int(count)

    out = {}
    for mod_name in active_modality_names:
        offset, count = modality_offsets[mod_name]
        idx = []
        for df_idx in range(num_df_local):
            base = df_idx * tokens_per_df
            idx.extend(range(base + offset, base + offset + count))
        out[mod_name] = np.array(idx, dtype=np.int64)

    return out, layout_local, num_df_local


def maybe_subsample(indices, max_points=None):
    if max_points is None or len(indices) <= max_points:
        return indices
    keep = np.linspace(0, len(indices) - 1, max_points).round().astype(np.int64)
    return indices[keep]


def coords_for_indices(rope_pos_1b, indices_np):
    """
    Convert rope_pos [1, N, 3] with coord order [t, x, y]
    into plotting coords [x, y, t].
    """
    coords = rope_pos_1b[0, indices_np, :].detach().cpu().numpy()
    xyz = np.stack([coords[:, 1], coords[:, 2], coords[:, 0]], axis=-1)
    return xyz


def set_equal_box(ax, xyz_list):
    valid = [x for x in xyz_list if x is not None and len(x) > 0]
    all_xyz = np.concatenate(valid, axis=0)
    mins = all_xyz.min(axis=0)
    maxs = all_xyz.max(axis=0)
    centers = (mins + maxs) / 2.0
    radii = (maxs - mins) / 2.0
    radius = float(np.max(radii))
    if radius <= 0:
        radius = 1.0

    ax.set_xlim(centers[0] - radius, centers[0] + radius)
    ax.set_ylim(centers[1] - radius, centers[1] + radius)
    ax.set_zlim(centers[2] - radius, centers[2] + radius)
    ax.set_box_aspect((1, 1, 1))


def modality_color_map(active_modality_names):
    cmap = plt.get_cmap("tab10")
    return {mod_name: cmap(i % 10) for i, mod_name in enumerate(active_modality_names)}


def plot_modality_colored_3d(
    ax,
    rope_pos,
    modality_indices_map,
    colors,
    title,
    max_points_per_modality=None,
    elev=24,
    azim=-63,
    point_size=18,
    point_alpha=0.95,
    line_alpha=0.8,
    linewidth=1.6,
    show_lines=True,
    show_points=True,
):
    xyz_all = []

    for mod_name, idx_full in modality_indices_map.items():
        idx = maybe_subsample(idx_full, max_points=max_points_per_modality)
        if len(idx) == 0:
            continue

        xyz = coords_for_indices(rope_pos, idx)
        xyz_all.append(xyz)
        color = colors[mod_name]

        if show_lines:
            ax.plot(
                xyz[:, 0],
                xyz[:, 1],
                xyz[:, 2],
                linewidth=linewidth,
                alpha=line_alpha,
                color=color,
                label=mod_name,
            )
        if show_points:
            ax.scatter(
                xyz[:, 0],
                xyz[:, 1],
                xyz[:, 2],
                s=point_size,
                alpha=point_alpha,
                depthshade=False,
                color=color,
            )

    ax.set_title(title, fontsize=14, weight="bold")
    ax.set_xlabel("x axis", fontsize=12)
    ax.set_ylabel("y axis", fontsize=12)
    ax.set_zlabel("t axis", fontsize=12)
    ax.view_init(elev=elev, azim=azim)
    ax.legend(loc="upper left", bbox_to_anchor=(0.0, 1.0), fontsize=9)

    return xyz_all


def print_per_modality_examples(
    rope_pos_current,
    rope_pos_diag,
    modality_indices_map,
    k=6,
):
    print("Per-modality coord samples")
    print("  format shown below is [t, x, y]")
    print()

    for mod_name, idx in modality_indices_map.items():
        if len(idx) == 0:
            continue
        k_eff = min(k, len(idx))
        idx_k = idx[:k_eff]

        cur = rope_pos_current[0, idx_k, :].detach().cpu()
        diag = rope_pos_diag[0, idx_k, :].detach().cpu()

        print(f"[{mod_name}] first {k_eff} global token indices:")
        print(idx_k.tolist())
        print(f"[{mod_name}] current RoPE-input coords:")
        print(cur)
        print(f"[{mod_name}] diagonal-override coords:")
        print(diag)
        print()


# ----------------------------
# Build coords
# ----------------------------
rope_pos_current, rope_pos_diag = make_diagonal_layout_rope_pos_from_current(decoder_local, mmio_local)

modality_indices_map, layout_local, num_df_local = get_modality_indices_map(decoder_local)
active_modality_names = list(modality_indices_map.keys())
colors = modality_color_map(active_modality_names)

# ----------------------------
# Plot
# ----------------------------
fig = plt.figure(figsize=LOCAL_3D_POS_CONFIG["figsize"], dpi=LOCAL_3D_POS_CONFIG["dpi"])
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax2 = fig.add_subplot(1, 2, 2, projection="3d")

xyz_current = plot_modality_colored_3d(
    ax1,
    rope_pos_current,
    modality_indices_map,
    colors,
    title="(a) Current coord layout (actual RoPE input)",
    max_points_per_modality=LOCAL_3D_POS_CONFIG["max_points_per_modality"],
    elev=LOCAL_3D_POS_CONFIG["elev"],
    azim=LOCAL_3D_POS_CONFIG["azim"],
    point_size=LOCAL_3D_POS_CONFIG["point_size"],
    point_alpha=LOCAL_3D_POS_CONFIG["point_alpha"],
    line_alpha=LOCAL_3D_POS_CONFIG["line_alpha"],
    linewidth=LOCAL_3D_POS_CONFIG["linewidth"],
    show_lines=LOCAL_3D_POS_CONFIG["show_lines"],
    show_points=LOCAL_3D_POS_CONFIG["show_points"],
)

xyz_diag = plot_modality_colored_3d(
    ax2,
    rope_pos_diag,
    modality_indices_map,
    colors,
    title="(b) Notebook diagonal layout override",
    max_points_per_modality=LOCAL_3D_POS_CONFIG["max_points_per_modality"],
    elev=LOCAL_3D_POS_CONFIG["elev"],
    azim=LOCAL_3D_POS_CONFIG["azim"],
    point_size=LOCAL_3D_POS_CONFIG["point_size"],
    point_alpha=LOCAL_3D_POS_CONFIG["point_alpha"],
    line_alpha=LOCAL_3D_POS_CONFIG["line_alpha"],
    linewidth=LOCAL_3D_POS_CONFIG["linewidth"],
    show_lines=LOCAL_3D_POS_CONFIG["show_lines"],
    show_points=LOCAL_3D_POS_CONFIG["show_points"],
)

set_equal_box(ax1, xyz_current + xyz_diag)
set_equal_box(ax2, xyz_current + xyz_diag)

fig.suptitle(
    "3D position comparison by modality\n"
    f"rope_temporal_scale_factor={viz_cfg_local['rope_temporal_scale_factor']}",
    fontsize=15,
    weight="bold",
    y=0.98,
)

plt.tight_layout()
plt.show()

# ----------------------------
# Numeric sanity logs
# ----------------------------
print("3D position plot summary")
print(f"  rope_temporal_scale_factor requested in notebook: {viz_cfg_local['rope_temporal_scale_factor']}")
print("  current panel uses decoder_local['target_rope_pos'] directly")
print("  so the first column below should already reflect the temporal scaling seen by RoPE")
print()

print_per_modality_examples(
    rope_pos_current=rope_pos_current,
    rope_pos_diag=rope_pos_diag,
    modality_indices_map=modality_indices_map,
    k=int(LOCAL_3D_POS_CONFIG["points_per_modality_to_log"]),
)